<a href="https://colab.research.google.com/github/Hanna07111/masked-social-signals/blob/review-notes/transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### encode
- task 별로 encoding 진행
- gaze, headpose, pose => vqvae
- speaker, bite => 프레임의 speaking 여부에 따라 라벨링 및 임베딩
- word => LinearNet 사용

### decode
- task 별로 decoding 진행
- gaze, headpose, pose => vqvae
- speaker, bite => speaekr_classifier, bite_classifier (LinearNet) 사용

In [ ]:
 def decode(self, output, task):

        # 디코딩 이전 표준화 과정
        task_idx = self.task_list.index(task)
        # 모달리티 슬라이싱
        task_output = output[:, task_idx::len(self.task_list), :]

        # 모달리티 별 제각각인 차원을 맞춰줌
        if task in self.padding_list:
            if self.feature_filling == 'pad':
                task_output = task_output[:, :, :128]
            elif self.feature_filling == 'repeat':
                task_output = task_output.reshape(task_output.size(0), task_output.size(1), -1, 128).mean(dim=2)

        task_output_reshaped = task_output.contiguous().view(self.bz*3*self.segment, -1)

        if task == 'gaze':
            #task_output_reshaped = self.gaze_projector(task_output_reshaped)
            if self.training:
                return task_output_reshaped
            _, x_hat, _ = self.gaze_vqvae.decode(task_output_reshaped, hard=True)
            return x_hat.view(self.bz, 3, self.segment*self.segment_length, -1)

        elif task == 'headpose':
            #task_output_reshaped = self.headpose_projector(task_output_reshaped)
            if self.training:
                return task_output_reshaped
            _, x_hat, _ = self.headpose_vqvae.decode(task_output_reshaped, hard=True)
            return x_hat.view(self.bz, 3, self.segment*self.segment_length, -1)

        elif task == 'pose':
            #task_output_reshaped = self.pose_projector(task_output_reshaped)
            if self.training:
                return task_output_reshaped
            _, x_hat, _ = self.pose_vqvae.decode(task_output_reshaped, hard=True)
            return x_hat.view(self.bz, 3, self.segment*self.segment_length//2, -1)

        elif task == 'speaker':
            return self.speaker_classifier(task_output_reshaped).view(self.bz, 3, self.segment, -1)

        elif task == 'bite':
            return self.bite_classifier(task_output_reshaped).view(self.bz, 3, self.segment, -1)

        return None

### forward

1. 데이터 증강 -> 사람 순서 셔플
2. 모달리티 별 인코딩 -> padding 도 여기서
3. 교차로 시퀀스 만들기
4. 임베딩 추가
5. 마스킹
6. 결과값 예측 및 저장

In [1]:
def forward(self, batch):
        self.bz = batch['gaze'].size(0)

        encode_list = []
        ys = []

        # data augmentation
        # 데이터 증강 -> 사람 순서 셔플 (사람 1은 항상 같은 위치.. 같은 편향 못 배우게)
        if self.training:
            random_sequences = torch.tensor([[0,1,2], [1,2,0], [2,0,1]])
            shuffled_people = random_sequences[torch.randint(0,3,(1,))].squeeze(0)
            batch = {k: v[:, shuffled_people, ...] for k, v in batch.items()}
            self.person_cyclic_encoding.person_encoding = self.person_cyclic_encoding.person_encoding[shuffled_people]

        # encode all the tasks
        # 모달리티 별 인코딩
        for task in self.task_list:
            current = batch[task]
            y, encode = self.encode(current, task)
            # 차원 맞춰주기
            encode_padded = self.padding(encode, task)


            encode_list.append(encode_padded)

            if task in ['speaker', 'bite']:
                ys.append(y)

        # 교차로 시퀀스 만들기 (시간 블록이 최우선, 그 안에서 사람X모달 교차될 수 있게)
        # it will make the input as (gaze_p1_t1, headpose_p1_t1, pose_p1_t1, word_p1_t1, gaze_p2_t1, headpose_p2_t1, pose_p2_t1, wordp2_t1, ...)
        stacked_inputs = torch.stack(encode_list, dim=3).permute(0, 2, 1, 3, 4) # (bz, 12, 3, 6, 1024)

        # segment, person, feature embedding (위치, 사람, 모달리티 정보)
        # add segment embeddings
        stacked_inputs = self.add_time_embedding(stacked_inputs)

        # add person encoding
        stacked_inputs = self.person_cyclic_encoding(stacked_inputs)

        # add feature embeddings
        stacked_inputs = self.add_feature_embedding(stacked_inputs)


        # 일부 토큰 마스킹 -> 일부러 정보 일부가 빠진 noisy input을 줘서 모델이 주변 시그널로 의미 있는 패턴을 잡도록 유도
        # mask some of the inputs
        stacked_inputs = self.mask_feature(stacked_inputs)

        stacked_inputs_reshaped = stacked_inputs.reshape(self.bz, -1, self.hidden_size)

        output = self.transformer(inputs_embeds=stacked_inputs_reshaped)['last_hidden_state'] # (self.bz, 12*3*6, 64)

        output = output.view(stacked_inputs.size()).permute(0, 2, 1, 3, 4).reshape(self.bz*3, -1, self.hidden_size)

        y_hats = []

        # output 이용해 결과값 예측
        # decode speaker and bite tasks
        for task in self.task_list:
            if task in ['speaker', 'bite']:
                y_hat = self.decode(output, task)
                y_hats.append(y_hat)

        return ys, y_hats